Create the base list of the repos from URLs

In [10]:
import pandas as pd
import os

# === CONFIGURATION ===
INPUT_CSV = r"D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone\Sorted_URL_List.csv"
OUTPUT_CSV = r"D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone\Analysis Output\7.4-Total_Repos.csv"

# === LOAD CSV ===
df = pd.read_csv(INPUT_CSV)

# === EXTRACT COMPONENTS IN LOWERCASE ===
df['clone_url'] = df['clone_url'].astype(str)
df['username'] = df['clone_url'].apply(lambda x: x.split('/')[-2].lower())

# ✅ Robust: get last path part and remove trailing '.git' only
df['project_name'] = df['clone_url'].apply(
    lambda x: x.split('/')[-1].lower().removesuffix('.git')
)

# ✅ Combine for full name with double underscore
df['full_name'] = df['username'] + '__' + df['project_name']

# Reset index to make it a column
df = df.reset_index()

# === REORDER: index + clone_url first ===
df_clean = df[['index', 'clone_url', 'username', 'project_name', 'full_name']]
df_clean.to_csv(OUTPUT_CSV, index=False)

print(f"✅ Cleaned repo list saved to: {OUTPUT_CSV}")


✅ Cleaned repo list saved to: D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone\Analysis Output\7.4-Total_Repos.csv


Aggregate YMLs and Builds

In [11]:
import pandas as pd
import os

# === CONFIGURATION ===
base_dir = r"D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone\Analysis Output"

# === FILE PATHS ===
yml_csv = os.path.join(base_dir, "7.1-YML_List_4th_attempt_ShallowC.csv")
yml_output_csv = os.path.join(base_dir, "7.1-YML_List_4th_attempt_ShallowC_Aggregated.csv")

gradle_csv = os.path.join(base_dir, "7.1-Gradle_List_4th_attempt_ShallowC.csv")
gradle_output_csv = os.path.join(base_dir, "7.1-Gradle_List_4th_attempt_ShallowC_BUILD.csv")

# === LOAD ===
df_yml = pd.read_csv(yml_csv)
df_gradle = pd.read_csv(gradle_csv)

# === AGGREGATE YML ===
df_yml_agg = df_yml.groupby('full_name').agg({
    'ci_platform': lambda x: ', '.join(sorted(set(filter(pd.notna, x)))),
    'test_type': lambda x: ', '.join(sorted(set(filter(pd.notna, x)))),
    'matched_keywords': lambda x: ', '.join(sorted(set(filter(pd.notna, x)))),
    'unit_test': 'any',
    'instrumentation_test': 'any',
    'full_name': 'count'  # temp for counting rows
}).rename(columns={
    'unit_test': 'unit_test_ci',
    'instrumentation_test': 'instr_test_ci',
    'ci_platform': 'ci_platform_yml',
    'full_name': 'NBR_YAML'
}).reset_index()

# === EXPORT YML ===
df_yml_agg.to_csv(yml_output_csv, index=False)
print(f"✅ Aggregated YML file saved to: {yml_output_csv}")

# === AGGREGATE GRADLE (including ci_platform_build) ===
df_gradle_agg = df_gradle.groupby('full_name').agg({
    'has_local_unit_test': 'any',
    'has_local_instrumentation_test': 'any',
    'ci_platform_build': lambda x: ', '.join(sorted(set(filter(pd.notna, x)))),
    'full_name': 'count'  # for counting rows
}).rename(columns={
    'has_local_unit_test': 'unit_test_local',
    'has_local_instrumentation_test': 'instr_test_local',
    'ci_platform_build': 'ci_platform_build',
    'full_name': 'NBR_GRADLE'
}).reset_index()

# === EXPORT GRADLE ===
df_gradle_agg.to_csv(gradle_output_csv, index=False)
print(f"✅ Aggregated Gradle BUILD file saved to: {gradle_output_csv}")


✅ Aggregated YML file saved to: D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone\Analysis Output\7.1-YML_List_4th_attempt_ShallowC_Aggregated.csv
✅ Aggregated Gradle BUILD file saved to: D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone\Analysis Output\7.1-Gradle_List_4th_attempt_ShallowC_BUILD.csv


Merge 7.1 YML and Build to 7.4

In [4]:
import pandas as pd
import os

# === CONFIG ===
base_dir = r"D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone\Analysis Output"

# ✅ Make sure this matches your true base file name:
base_csv = os.path.join(base_dir, "7.4-Total_Repos.csv")  # use your actual name
yml_agg_csv = os.path.join(base_dir, "7.1-YML_List_4th_attempt_ShallowC_Aggregated.csv")
gradle_agg_csv = os.path.join(base_dir, "7.1-Gradle_List_4th_attempt_ShallowC_BUILD.csv")
output_csv = os.path.join(base_dir, "7.4-Total_Repos_YML_Merged.csv")

# === LOAD ===
df_base = pd.read_csv(base_csv)
df_yml_agg = pd.read_csv(yml_agg_csv)
df_gradle_agg = pd.read_csv(gradle_agg_csv)

# === NORMALIZE KEYS ===
df_base['full_name'] = df_base['full_name'].astype(str).str.strip().str.lower()
df_yml_agg['full_name'] = df_yml_agg['full_name'].astype(str).str.strip().str.lower()
df_gradle_agg['full_name'] = df_gradle_agg['full_name'].astype(str).str.strip().str.lower()

# === MERGE BASE + YAML ===
df_merged = df_base.merge(
    df_yml_agg[['full_name', 'ci_platform_yml', 'test_type', 'matched_keywords', 'unit_test_ci', 'instr_test_ci', 'NBR_YAML']],
    on='full_name',
    how='left'
)

# === MERGE RESULT + GRADLE ===
df_merged = df_merged.merge(
    df_gradle_agg[['full_name', 'unit_test_local', 'instr_test_local', 'ci_platform_build', 'NBR_GRADLE']],
    on='full_name',
    how='left'
)

# === SAVE ===
df_merged.to_csv(output_csv, index=False)
print(f"✅ Final merged file saved to: {output_csv}")

print(f"✅ Base rows: {len(df_base)}")
print(f"✅ YAML agg rows: {len(df_yml_agg)}")
print(f"✅ Gradle agg rows: {len(df_gradle_agg)}")
print(f"✅ Rows with YAML info: {df_merged['ci_platform_yml'].notna().sum()}")
print(f"✅ Rows with Gradle info: {df_merged['unit_test_local'].notna().sum()}")


✅ Final merged file saved to: D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone\Analysis Output\7.4-Total_Repos_YML_Merged.csv
✅ Base rows: 2614
✅ YAML agg rows: 120
✅ Gradle agg rows: 125
✅ Rows with YAML info: 120
✅ Rows with Gradle info: 125


In [12]:
import pandas as pd
import os

# === CONFIG ===
base_dir = r"D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone\Analysis Output"

# ✅ Adjust these filenames to match your actual files in the folder:
base_csv = os.path.join(base_dir, "7.4-Total_Repos.csv")  
yml_agg_csv = os.path.join(base_dir, "7.1-YML_List_4th_attempt_ShallowC_Aggregated.csv")
gradle_agg_csv = os.path.join(base_dir, "7.1-Gradle_List_4th_attempt_ShallowC_BUILD.csv")
api_agg_csv = os.path.join(base_dir, "7.2-Project_List_API_4thAttempt_ShallowC.csv")  # your new API columns
output_csv = os.path.join(base_dir, "7.4-Total_Repos_YML_Merged.csv")

# === LOAD ALL CSVs ===
print("🔹 Loading files...")
df_base = pd.read_csv(base_csv)
df_yml_agg = pd.read_csv(yml_agg_csv)
df_gradle_agg = pd.read_csv(gradle_agg_csv)
df_api_agg = pd.read_csv(api_agg_csv)

# === NORMALIZE 'full_name' in ALL ===
for df in [df_base, df_yml_agg, df_gradle_agg, df_api_agg]:
    df['full_name'] = df['full_name'].astype(str).str.strip().str.lower().str.replace('/', '__')

# === MERGE: BASE + YAML ===
df_merged = df_base.merge(
    df_yml_agg[['full_name', 'ci_platform_yml', 'test_type', 'matched_keywords',
                'unit_test_ci', 'instr_test_ci', 'NBR_YAML']],
    on='full_name',
    how='left'
)

# === MERGE: + GRADLE ===
df_merged = df_merged.merge(
    df_gradle_agg[['full_name', 'unit_test_local', 'instr_test_local',
                   'ci_platform_build', 'NBR_GRADLE']],
    on='full_name',
    how='left'
)

# === MERGE: + API AGGREGATED (NEW) ===
df_merged = df_merged.merge(
    df_api_agg[['full_name', 'distinct_api_levels', 'yml_count', 'yaml_errors']],
    on='full_name',
    how='left'
)

# === SAVE FINAL ===
df_merged.to_csv(output_csv, index=False)
print(f"✅ Final merged file saved to: {output_csv}")

# === PRINT SUMMARY ===
print("\n🔑 Merge Summary:")
print(f"🔹 Base rows: {len(df_base)}")
print(f"🔹 YAML agg rows: {len(df_yml_agg)}")
print(f"🔹 Gradle agg rows: {len(df_gradle_agg)}")
print(f"🔹 API agg rows: {len(df_api_agg)}")
print(f"✅ Rows with YAML info: {df_merged['ci_platform_yml'].notna().sum()}")
print(f"✅ Rows with Gradle info: {df_merged['unit_test_local'].notna().sum()}")
print(f"✅ Rows with API info: {df_merged['distinct_api_levels'].notna().sum()}")


🔹 Loading files...
✅ Final merged file saved to: D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone\Analysis Output\7.4-Total_Repos_YML_Merged.csv

🔑 Merge Summary:
🔹 Base rows: 2614
🔹 YAML agg rows: 120
🔹 Gradle agg rows: 125
🔹 API agg rows: 120
✅ Rows with YAML info: 120
✅ Rows with Gradle info: 125
✅ Rows with API info: 120


Create number of Contributors and Number of Commits files

In [13]:
import os
import pandas as pd

# === CONFIG ===
BASE_DIR = r"D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone"
COMMITS_DIR = os.path.join(BASE_DIR, "Commits")
OUTPUT_DIR = os.path.join(BASE_DIR, "Analysis Output")

os.makedirs(OUTPUT_DIR, exist_ok=True)

commits_data = []
contributors_data = []

for file_name in os.listdir(COMMITS_DIR):
    file_path = os.path.join(COMMITS_DIR, file_name)

    # === Extract between first and third "__" ===
    parts = file_name.split("__")
    if len(parts) >= 3:
        raw_full_name = parts[1] + "__" + parts[2]
        # === Normalize: strip + lowercase ===
        full_name = raw_full_name.strip().lower()
    else:
        print(f"Skipping file with unexpected format: {file_name}")
        continue

    if file_name.endswith(".csv"):
        try:
            df = pd.read_csv(file_path)
            num_commits = len(df)
            commits_data.append({"full_name": full_name, "number_of_commits": num_commits})
        except Exception as e:
            print(f"Error reading commits CSV {file_name}: {e}")

    elif file_name.endswith(".txt"):
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                contributors = [line.strip() for line in f if line.strip()]
                unique_contributors = set(contributors)
                num_contributors = len(unique_contributors)
                contributors_data.append({"full_name": full_name, "number_of_contributors": num_contributors})
        except Exception as e:
            print(f"Error reading contributors TXT {file_name}: {e}")

# === Save commits count ===
df_commits = pd.DataFrame(commits_data)
commits_output = os.path.join(OUTPUT_DIR, "7.4-number_of_commits.csv")
df_commits.to_csv(commits_output, index=False)
print(f"✅ Saved: {commits_output}")

# === Save contributors count ===
df_contributors = pd.DataFrame(contributors_data)
contributors_output = os.path.join(OUTPUT_DIR, "7.4-number_of_contributors.csv")
df_contributors.to_csv(contributors_output, index=False)
print(f"✅ Saved: {contributors_output}")


✅ Saved: D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone\Analysis Output\7.4-number_of_commits.csv
✅ Saved: D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone\Analysis Output\7.4-number_of_contributors.csv


appending metadata

In [14]:
import pandas as pd
import os

# === CONFIG ===
base_dir = r"D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone"
output_dir = os.path.join(base_dir, "Analysis Output")

metadata_csv = os.path.join(base_dir, "8.2-Project_Metadata.csv")
base_final_csv = os.path.join(output_dir, "7.4-Total_Repos_YML_Merged.csv")
output_final_csv = os.path.join(output_dir, "7.4-Total_Repos_YML_Merged_Metadata.csv")

# === LOAD ===
df_meta = pd.read_csv(metadata_csv)
df_base = pd.read_csv(base_final_csv)

# === RENAME AND CREATE NORMALIZED FULL NAME ===
df_meta = df_meta.rename(columns={'full_name': 'full_name_metadata'})
df_meta['full_name'] = df_meta['full_name_metadata'].astype(str).str.replace("/", "__")
df_meta['full_name'] = df_meta['full_name'].str.strip().str.lower()

# === NORMALIZE BASE ===
df_base['full_name'] = df_base['full_name'].astype(str).str.strip().str.lower()

# === SELECT METADATA COLUMNS ===
metadata_columns = [
    'full_name', 'language', 'license', 'created_at', 'updated_at', 'last_commit_date',
    'stars', 'forks', 'watchers', 'open_issues', 'size'
]

df_meta_selected = df_meta[metadata_columns]

# === MERGE ===
df_merged = df_base.merge(
    df_meta_selected,
    on='full_name',
    how='left'
)

# === SAVE ===
df_merged.to_csv(output_final_csv, index=False)
print(f"✅ Final merged file with metadata saved to: {output_final_csv}")
print(f"✅ Base rows: {len(df_base)}")
print(f"✅ Metadata rows: {len(df_meta)}")
print(f"✅ Matched rows with metadata: {df_merged['language'].notna().sum()}")


✅ Final merged file with metadata saved to: D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone\Analysis Output\7.4-Total_Repos_YML_Merged_Metadata.csv
✅ Base rows: 2614
✅ Metadata rows: 138
✅ Matched rows with metadata: 138


adding the number of contributos and commits

In [15]:
import pandas as pd
import os

# === CONFIG ===
BASE_DIR = r"D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone\Analysis Output"

# File paths
main_file = os.path.join(BASE_DIR, "7.4-Total_Repos_YML_Merged_Metadata.csv")
commits_file = os.path.join(BASE_DIR, "7.4-number_of_commits.csv")
contributors_file = os.path.join(BASE_DIR, "7.4-number_of_contributors.csv")
output_file = os.path.join(BASE_DIR, "7.4-Total_Repos_YML_Merged_Metadata.csv")

# === Load files ===
df_main = pd.read_csv(main_file)
df_commits = pd.read_csv(commits_file)
df_contributors = pd.read_csv(contributors_file)

# === Left join on 'full_name' ===
df_joined = df_main.merge(df_commits, on='full_name', how='left')
df_joined = df_joined.merge(df_contributors, on='full_name', how='left')

# === Save the final result ===
df_joined.to_csv(output_file, index=False)

print(f"✅ Merged file saved to: {output_file}")


✅ Merged file saved to: D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone\Analysis Output\7.4-Total_Repos_YML_Merged_Metadata.csv


Creating General CI Platform and Unit / instr testing

In [17]:
import pandas as pd

# === 1) File paths ===
input_path = r"D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone\Analysis Output\7.4-Total_Repos_YML_Merged_Metadata.csv"
output_path = r"D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone\Analysis Output\7.4-Total_Repos_YML_Merged_Metadata_Updated.csv"

# === 2) Load CSV ===
df = pd.read_csv(input_path)

# === 3) Create 'CI Platform' ===
df['CI Platform'] = df[['ci_platform_yml', 'ci_platform_build']] \
    .fillna('') \
    .agg(','.join, axis=1) \
    .str.replace(r',+', ',', regex=True) \
    .str.strip(',')

# === 4) Use robust boolean logic without fillna ===
df['Unit_Test(CI or Local)'] = (df['unit_test_ci'] == True) | (df['unit_test_local'] == True)
df['Instr_Test(CI or Local)'] = (df['instr_test_ci'] == True) | (df['instr_test_local'] == True)

# === 5) Save ===
df.to_csv(output_path, index=False)
print(f"✅ Updated file saved to: {output_path}")


✅ Updated file saved to: D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone\Analysis Output\7.4-Total_Repos_YML_Merged_Metadata_Updated.csv
